# 05 — Watchlist Predictions

Predicts a rating for every unseen film on the watchlist, plus a "coming soon" set of
unreleased films, for the Streamlit app.

**Input:** the Letterboxd export (`watchlist.csv`, via `.env`), `data/interim/modelling_base.csv`
(the rated history) and `data/interim/films_enriched.csv` (rated films' TMDB data)
**Output** (all in `data/processed/`, read by the app): `watchlist_predictions.csv` (3,924 films,
with the gap above the crowd), `neighbours.csv` (the 5 rated films behind each prediction) and
`coming_soon.csv` (29 films)

The watchlist goes through the same pipeline as the rated films — the same matching rules
(`src/tmdb.py`), the same features (`src/features.py`) — and is scored by the **deployable
model** from `04` §15, refitted on all 1,192 rated viewings. Unreleased films have no crowd
score, so they get the **no-crowd variant** from `04` §16 instead.

Every watchlist and upcoming-release API response is cached in its **own files**, so the
rated-film caches from `02` are never touched.

## 1. Setup

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from src.features import (PLAIN_NUMERIC, LOG_NUMERIC, HISTORY_KEYS,
                          genre_vocabulary, genre_history, top_n_vocabulary,
                          build_features_F, build_features_H,
                          keyword_scores, add_keyword_score)
from src.letterboxd import load_export, film_key
from src.tmdb import (make_headers, match_all, AUTO_ACCEPT, search_film,
                      fetch_all_details, discover_upcoming)

## 2. Load the watchlist

Keyed with the same `film_key` as the diary, so a watchlist film since rated would produce an
identical key and be caught.

In [2]:
load_dotenv()
export = load_export(os.getenv("LETTERBOXD_EXPORT_DIR"))

watchlist = export["watchlist"].copy()
watchlist["film_key"] = film_key(watchlist)

rated   = set(film_key(export["ratings"]))
watched = set(film_key(export["watched"]))

print(f"watchlist rows   : {len(watchlist)}")
print(f"unique film_keys : {watchlist['film_key'].nunique()}")
print(f"missing year     : {watchlist['Year'].isna().sum()}")
print(f"already rated    : {watchlist['film_key'].isin(rated).sum()}")
print(f"already watched  : {watchlist['film_key'].isin(watched).sum()}")

watchlist rows   : 4031
unique film_keys : 4022
missing year     : 8
already rated    : 0
already watched  : 0


Nothing overlaps with rated or watched films — Letterboxd removes films from the
watchlist once logged. But 4,031 rows give only 4,022 unique keys, and 8 films have no year.

In [3]:
dupes = watchlist[watchlist["film_key"].duplicated(keep=False)].sort_values("film_key")
print(f"{len(dupes)} rows share a film_key ({dupes['film_key'].nunique()} keys)\n")
print(dupes[["Name", "Year", "Letterboxd URI"]].to_string(index=False))

print("\nmissing year:\n")
print(watchlist[watchlist["Year"].isna()][["Name", "Letterboxd URI"]].to_string(index=False))

10 rows share a film_key (1 keys)

                    Name   Year        Letterboxd URI
              The Castle 1997.0  https://boxd.it/1sWI
              The Castle 1997.0  https://boxd.it/1Pho
                 Polaris    NaN  https://boxd.it/vMS2
       The Memory Police    NaN  https://boxd.it/scb6
         The Governesses    NaN  https://boxd.it/AdVq
           Tower Stories    NaN  https://boxd.it/nzTg
              Love Child    NaN  https://boxd.it/fAP4
The Bookie & the Bruiser    NaN  https://boxd.it/N6Oe
    Here Comes the Flood    NaN  https://boxd.it/qdjm
              Lily May B    NaN https://boxd.it/13ssm

missing year:

                    Name        Letterboxd URI
                 Polaris  https://boxd.it/vMS2
       The Memory Police  https://boxd.it/scb6
         The Governesses  https://boxd.it/AdVq
           Tower Stories  https://boxd.it/nzTg
              Love Child  https://boxd.it/fAP4
The Bookie & the Bruiser  https://boxd.it/N6Oe
    Here Comes the Flood  

**Undated films have no key at all.** A missing year makes `film_key` **NaN**, not a string —
`nunique` ignores NaN and `duplicated` treats NaNs as equal, which is why all 8 undated films
appear as one "shared key". They look like announced but unreleased films.

**One genuine collision.** *The Castle* (1997) appears twice with **different URIs** — Rob
Sitch's comedy and Haneke's Kafka adaptation. Title and year cannot separate them, so the
matcher would give both the same search results and the same TMDB film.

**Two pipeline rules follow:**

- **Undated films are excluded.** No key, no year filter for the search — exactly where
  title collisions win — and several features need a year anyway. 8 films (0.2%).
- **Key collisions are held back from automatic matching** and resolved by override,
  keyed by URI, below. Any export can contain these, so the rule is general.

`& ~undated` keeps the NaN keys from also counting as a collision, so each film falls under
exactly one rule. The URI is renamed to `film_uri` because `match_all` iterates with
`itertuples`, which cannot address a column name containing a space.

In [4]:
undated  = watchlist["film_key"].isna()
collides = watchlist["film_key"].duplicated(keep=False) & ~undated

collisions = watchlist[collides].copy()

films_wl = (watchlist[~undated & ~collides]
            .rename(columns={"Name": "film_title", "Year": "film_year",
                             "Letterboxd URI": "film_uri"})
            [["film_key", "film_title", "film_year", "film_uri"]]
            .reset_index(drop=True))

print(f"excluded, no year      : {undated.sum()}")
print(f"held back, collision   : {collides.sum()} rows, {collisions['film_key'].nunique()} key(s)")
print(f"to match automatically : {len(films_wl)}")

excluded, no year      : 8
held back, collision   : 2 rows, 1 key(s)
to match automatically : 4021


## 3. Matching

`match_all` from `02`, unchanged, with its own cache file. 4,021 new calls on the first run
(cached since).

In [5]:
HEADERS = make_headers(os.getenv("TMDB_TOKEN"))

WL_SEARCH_CACHE = Path("data/cache/watchlist_search_raw.json")

wl_matches = match_all(films_wl, headers=HEADERS, cache_path=WL_SEARCH_CACHE)

done — 0 new API calls, 4021 from cache


In [6]:
print(f"films: {len(wl_matches)}\n")
print("confidence breakdown")
for tier, n in wl_matches["confidence"].value_counts().items():
    print(f"  {tier:12s} {n:5d}  ({n/len(wl_matches)*100:5.1f}%)")

wl_review = wl_matches[~wl_matches["confidence"].isin(AUTO_ACCEPT)]
print(f"\nauto-accepted  : {len(wl_matches) - len(wl_review)} "
      f"({(len(wl_matches) - len(wl_review)) / len(wl_matches) * 100:.1f}%)")
print(f"needs review   : {len(wl_review)}")
print(f"no match at all: {wl_matches['tmdb_id'].isna().sum()}")

films: 4021

confidence breakdown
  exact         3673  ( 91.3%)
  year_off       295  (  7.3%)
  weak            49  (  1.2%)
  close            3  (  0.1%)
  no_match         1  (  0.0%)

auto-accepted  : 3968 (98.7%)
needs review   : 53
no match at all: 1


**98.7% auto-accepted**, against 99.1% for the rated films. `year_off` is higher (7.3% vs
5.9%), consistent with more festival and recent titles, where premiere and release years
differ. **53 need review and 1 has no match.**

In [7]:
print(wl_review.sort_values(["confidence", "similarity"])[
    ["film_title", "film_year", "matched_title", "matched_year",
     "confidence", "similarity", "vote_count"]
].to_string(index=False))

                                             film_title  film_year                                           matched_title  matched_year confidence  similarity  vote_count
                                                Monster     2018.0                                                 Monster        2018.0      close       1.000         1.0
                                                  Alpha     2025.0                                                   Alpha        2025.0      close       1.000       140.0
                                                Solaris     2007.0                                                 Solaris        2007.0      close       1.000         1.0
   Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles     1975.0                                                     NaN           NaN   no_match       0.000         NaN
                                  Q: The Winged Serpent     1982.0                                                       Q        1982.0    

### Resolving the 53

Sorted by tier then similarity so the weakest come first; `vote_count` added because at this
scale it helps separate *correct but penalised* (high similarity, or a well-known film) from
*genuinely wrong* (low similarity and few votes).

They fall into four groups:

- **Accept as matched — 37.** Title variants (*Q*, *Day of the Woman*, *The World of Apu*,
  *2010*, *Nine 1/2 Weeks*, *Pusher 3* …), year gaps over one year (premiere vs release again,
  stretched further), and two checked below — *Nausicaä* and *The Civil War on Drugs*.
  *Alpha* (2025) is correct but was demoted because the 2018 *Alpha* has far more votes: the
  demotion rule's "best-known rival" check ignores year.
- **Exclude as television — 10.** *The New Pope*, *The Young Pope*, *Ren Faire*, *Storm of the
  Century*, *Berlin Alexanderplatz*, *Dekalog*, *Salem's Lot* (1979), *Little Women* (2017),
  *Catch-22* (2019), *Sybil* (1976) — each matched to an unrelated film or a namesake.
- **Exclude as not on TMDB — 1.** *The Murmuring* (2022).
- **Override — 5.** *Jeanne Dielman*, *Precious*, *The Ear*, *The Night*, *Monster*: wrong film
  matched, real one exists.

The IDs come from the searches below and from **each film's Letterboxd page, which links to
its TMDB record** — the ID Letterboxd itself uses, so no guessing on either side.

In [8]:
lookups = {
    "Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)": "Jeanne Dielman",
    "Precious (2009)":                           "Precious",
    "The Ear (1970)":                            "The Ear",
    "The Night (2021)":                          "The Night",
    "Monster (2018)":                            "Monster",
    "Nausicaä of the Valley of the Wind (1984)": "Nausicaä of the Valley of the Wind",
    "The Murmuring (2022)":                      "The Murmuring",
    "The Civil War on Drugs (2011)":             "The Civil War on Drugs",
    "Sybil (1976)":                              "Sybil",
    "The Castle (1997)":                         "The Castle",
}

current = wl_matches.set_index("film_key")["tmdb_id"]

for key, query in lookups.items():
    year = int(key[-5:-1])
    print(f"=== {key}   current match: {current.get(key, 'held back')}")
    for r in search_film(query, headers=HEADERS):
        release = r.get("release_date") or ""
        y = int(release[:4]) if release[:4].isdigit() else None
        if y is None or abs(y - year) <= 5:
            print(f"  {r['id']:>8}  {y or '????'}  {r.get('vote_count', 0):>6} votes  "
                  f"{r['title']}  /  {r['original_title']}")
    print()

=== Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)   current match: nan
    308191  1975       2 votes  Around Jeanne Dielman  /  Autour de Jeanne Dielman
     44012  1976     410 votes  Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles  /  Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles
   1404470  ????       0 votes  Exercises in Style. Jeanne Dielman  /  Exercises in Style. Jeanne Dielman

=== Precious (2009)   current match: 1772671.0
     25793  2009    1821 votes  Precious: Based on the Novel 'Push' by Sapphire  /  Precious: Based on the Novel 'Push' by Sapphire
   1010417  ????       0 votes  Precious Cargo  /  Precious Cargo

=== The Ear (1970)   current match: 888896.0
    888896  1970       0 votes  The Burning Ear  /  The Burning Ear
    340000  1970       1 votes  The Eye Hears, the Ear Sees  /  The Eye Hears, the Ear Sees
    799285  ????       0 votes  Into the Ear, Everybody  /  Into the Ear, Everybody

=== The Night (2021)   current match: 604360.0


Settled here: *Jeanne Dielman* → 44012 (TMDB dates it 1976, so the 1975 search missed it),
*Precious* → 25793 (listed under the long title), *Nausicaä* is correct (ID 81 is the main film;
TMDB returns its English title as *Warriors of the Wind*), and *Sybil* (1976) does not appear in
the movie index at all — excluded as television. *The Castle*'s two films are 13852 (Sitch) and
26891 (Haneke).

Three needed a second search: *The Ear* was released in 1990 after being banned, so it is
searched by its Czech title; *The Night* and *Monster* are generic titles that rank below
TMDB's top 20.

In [9]:
second = [
    ("Ucho",      None),   # The Ear's original Czech title
    ("The Night", 2020),
    ("The Night", 2021),
    ("Monster",   2021),
    ("Monster",   2018),
]

for query, year in second:
    print(f"=== {query!r}, year={year}")
    for r in search_film(query, year, headers=HEADERS)[:8]:
        release = r.get("release_date") or "????"
        print(f"  {r['id']:>8}  {release[:4]}  {r.get('vote_count', 0):>6} votes  "
              f"{r['title']}  /  {r['original_title']}")
    print()

=== 'Ucho', year=None
     88953  1990      74 votes  The Ear  /  Ucho
    334772  1945       8 votes  The Eye & the Ear  /  Oko I Ucho
   1276412  2016       0 votes  The Internal Ear  /  Ucho wewnętrzne
   1064567  1972       0 votes  Kým sa ucho neodbije  /  Kým sa ucho neodbije
     64711  1949     142 votes  Long-Haired Hare  /  Long-Haired Hare
     63899  1977      57 votes  White Bim Black Ear  /  Белый Бим Чёрное ухо
    588336  2016       2 votes  The Mystery of Van Gogh's Ear  /  The Mystery of Van Gogh's Ear
     19316  2004     275 votes  Saving Face  /  Saving Face

=== 'The Night', year=2020
      3112  1955    1905 votes  The Night of the Hunter  /  The Night of the Hunter
    547565  2021    1392 votes  The Night House  /  The Night House
    565743  2019    1239 votes  The Vast of Night  /  The Vast of Night
    526007  2020     932 votes  The Night Clerk  /  The Night Clerk
    640796  2020       3 votes  Into the Night  /  Into the Night
    686245  2020     266 vot

*The Ear* → **88953**. *The Night* never surfaces — TMDB's `year` parameter boosts rather than
filters (*Night of the Hunter* appears for 2020) — so its ID, **854531**, came from its
Letterboxd page, as did confirmation of *Monster* → **489932**, *The Civil War on Drugs* (180988,
correct), *The Murmuring* (not on TMDB) and the *Castle* URIs (`1sWI` is Haneke).

### Applying the decisions

Recorded as data, as in `02`, so they are visible and reusable; `WL_EXCLUDE` records the
reason. Two safeguards: a warning if any key fails to match exactly, and an `assert` that both
*Castle* URIs map to an ID.

In [10]:
WL_OVERRIDES = {
    "Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)": 44012,
    "Precious (2009)":  25793,
    "The Ear (1970)":   88953,
    "The Night (2021)": 854531,
    "Monster (2018)":   489932,
}

WL_EXCLUDE = {
    "The New Pope (2020)":          "tv",
    "The Young Pope (2016)":        "tv",
    "Ren Faire (2024)":             "tv",
    "Storm of the Century (1999)":  "tv",
    "Berlin Alexanderplatz (1980)": "tv",
    "Dekalog (1989)":               "tv",
    "Salem's Lot (1979)":           "tv",
    "Little Women (2017)":          "tv",
    "Catch-22 (2019)":              "tv",
    "Sybil (1976)":                 "tv",
    "The Murmuring (2022)":         "not on TMDB",
}

COLLISION_OVERRIDES = {                 # by URI: film_key can't tell these apart
    "https://boxd.it/1sWI": 26891,      # The Castle — Haneke (Das Schloß)
    "https://boxd.it/1Pho": 13852,      # The Castle — Sitch
}

wl_final = wl_matches.merge(films_wl[["film_key", "film_uri"]], on="film_key", how="left")

for key in [*WL_OVERRIDES, *WL_EXCLUDE]:
    if key not in set(wl_final["film_key"]):
        print(f"warning: key not found — {key!r}")

for key, tmdb_id in WL_OVERRIDES.items():
    mask = wl_final["film_key"] == key
    wl_final.loc[mask, "tmdb_id"] = tmdb_id
    wl_final.loc[mask, "confidence"] = "manual_override"

before = len(wl_final)
wl_final = wl_final[~wl_final["film_key"].isin(WL_EXCLUDE)]
n_excluded = before - len(wl_final)

castle = pd.DataFrame({
    "film_key":   collisions["film_key"].values,
    "film_title": collisions["Name"].values,
    "film_year":  collisions["Year"].values,
    "film_uri":   collisions["Letterboxd URI"].values,
})
castle["tmdb_id"]    = castle["film_uri"].map(COLLISION_OVERRIDES)
castle["confidence"] = "manual_override"
assert castle["tmdb_id"].notna().all(), "a collision URI has no override"

wl_final = pd.concat([wl_final, castle], ignore_index=True)

print(f"overrides applied : {len(WL_OVERRIDES)} + {len(castle)} by URI")
print(f"excluded          : {n_excluded}")
print(f"final             : {len(wl_final)} films\n")
print(wl_final["confidence"].value_counts())
print(f"\nmissing tmdb_id   : {wl_final['tmdb_id'].isna().sum()}")
print(f"duplicate tmdb_id : {wl_final['tmdb_id'].duplicated().sum()}")

overrides applied : 5 + 2 by URI
excluded          : 11
final             : 4012 films

confidence
exact              3673
year_off            295
weak                 35
manual_override       7
close                 2
Name: count, dtype: int64

missing tmdb_id   : 0
duplicate tmdb_id : 0


**4,012 films**, no missing IDs, and **no duplicate IDs** — no two watchlist films point at the
same TMDB film, which is the tell-tale sign of a namesake match.

### Spot-check the `year_off` tier

295 films auto-accepted on a one-year gap. Same check as `02`, same rule set in advance: if
every sampled gap runs Letterboxd-earlier-than-TMDB and the films are genuine, the tier stands.

In [11]:
wl_final[wl_final["confidence"] == "year_off"][
    ["film_title", "film_year", "matched_title", "matched_year", "vote_count"]
].sample(10, random_state=0).to_string(index=False)

"                 film_title  film_year               matched_title  matched_year  vote_count\n                 The Bronze     2015.0                  The Bronze        2016.0       404.0\n       The Celluloid Closet     1995.0        The Celluloid Closet        1996.0       120.0\n          Heaven Knows What     2014.0           Heaven Knows What        2015.0       204.0\nAt the First Breath of Wind     2002.0 At the First Breath of Wind        2003.0        15.0\n            In Vanda's Room     2000.0             In Vanda's Room        2001.0        57.0\n             White Material     2009.0              White Material        2010.0       181.0\n                  Ned Rifle     2014.0                   Ned Rifle        2015.0        58.0\n            The Daytrippers     1996.0             The Daytrippers        1997.0       113.0\n           Imagine Me & You     2005.0            Imagine Me & You        2006.0      1136.0\n            35 Shots of Rum     2008.0             35 Shots

**The tier stands** — all ten run in the same direction and all are the intended films, down to
*At the First Breath of Wind* at 15 votes.

## 4. Details

`fetch_all_details` from `02`, with its own cache. `parse_details` now also keeps
**`poster_path`** — already in every cached response, so adding it cost no API calls.

In [12]:
WL_DETAILS_CACHE = Path("data/cache/watchlist_details.json")

wl_details = fetch_all_details(wl_final["tmdb_id"], headers=HEADERS, cache_path=WL_DETAILS_CACHE)
print(wl_details.shape)
print(f"missing poster: {wl_details['poster_path'].isna().sum()}")

done — 0 new API calls, 4012 from cache
(4012, 18)
missing poster: 0


Every watchlist film has a poster.

In [13]:
print("missing values:")
print(wl_details.isna().sum()[lambda s: s > 0])
print()
print(f"runtime = 0 or null : {((wl_details['runtime'] == 0) | wl_details['runtime'].isna()).sum()}")
print(f"no genres           : {(wl_details['genres'] == '').sum()}")
print(f"no keywords         : {(wl_details['keywords'] == '').sum()}")
print(f"no director         : {wl_details['director'].isna().sum()}")
print(f"no cinematographer  : {wl_details['cinematographer'].isna().sum()}")
print(f"zero vote_count     : {(wl_details['vote_count'] == 0).sum()}")

missing values:
collection_name    3568
cinematographer     100
dtype: int64

runtime = 0 or null : 4
no genres           : 1
no keywords         : 232
no director         : 0
no cinematographer  : 100
zero vote_count     : 20


Director is complete, and cinematographer gaps (100, 2.5%) go through Block H's usual
imputation. Two gaps interact with the model: **20 films have zero votes**, and TMDB reports
their crowd score as **0.0** — not a real score, and far below anything in training — and
**4 have a runtime of 0**, TMDB's "unknown". 232 have no keywords, a state the model barely
saw in training (0.6% of rated films); the keyword score falls back to a near-average value.

### Values outside the rated range

A Random Forest does not extrapolate — it treats any value beyond the training range like the
most extreme value it has seen. So the check is against the range of the 1,174 rated films,
which is what the final model is refitted on.

In [14]:
rated_films = pd.read_csv("data/interim/films_enriched.csv")

RANGE_COLS = ["runtime", "vote_average", "vote_count", "popularity"]
rated_lo, rated_hi = rated_films[RANGE_COLS].min(), rated_films[RANGE_COLS].max()
print(pd.DataFrame({"rated min": rated_lo, "rated max": rated_hi}), "\n")

outside = (wl_details[RANGE_COLS] < rated_lo) | (wl_details[RANGE_COLS] > rated_hi)
print("watchlist films outside the rated range, by column:")
print(outside.sum(), "\n")
print(f"outside on any column: {outside.any(axis=1).sum()}\n")

odd = (wl_details["runtime"].fillna(0) == 0) | (wl_details["genres"] == "") | (wl_details["vote_count"] == 0)
print(wl_details[odd][["tmdb_id", "tmdb_title", "release_date", "runtime", "genres",
                       "vote_count", "vote_average"]].to_string(index=False))

              rated min   rated max
runtime         45.0000    345.0000
vote_average     4.2910      8.6870
vote_count       7.0000  40184.0000
popularity       0.3222    690.8873 

watchlist films outside the rated range, by column:
runtime         65
vote_average    33
vote_count      45
popularity       2
dtype: int64 

outside on any column: 112

 tmdb_id                    tmdb_title release_date  runtime                                  genres  vote_count  vote_average
 1637225             My Brother Jordan   2020-08-19       63                             Documentary           0           0.0
  759517       Benighted but Not Begun   1994-01-01       24                                                   0           0.0
  634154                         Tiger   2011-10-31       70                                   Drama           0           0.0
  209124             How Far Is Heaven   2012-08-23       99                             Documentary           0           0.0
 1259211    

**112 films (2.8%)** fall outside on at least one column. The 20 zero-vote films explain every
odd value: all 4 zero runtimes and the one film with no genres are among them. **14 are
unreleased**, 2 are recent releases nobody has voted on yet (*Moonglow*, *Under the Same
Sun*) and 4 are obscure older films.

## 5. Prediction rules

Fixed **before any predictions existed**, so they cannot be tuned to make the list look good:

- **Minimum-data gate: fewer than 7 votes → excluded.** Below that the crowd score is missing
  or dominated by a handful of voters, and a 1-vote 10.0 would top the ranking. 7 is the
  lowest vote count among the rated films.
- **Shorts excluded: runtime of 40 minutes or less** — the Academy's definition. The viewer
  watches shorts but does not rate them (`01` dropped unrated entries, almost all shorts), so
  a predicted star rating for one predicts something that would never be produced.
- **Everything else still out of range is flagged, not excluded** — `out_of_range` names the
  offending columns, so the app can caveat the prediction rather than hide the film.
- **"Coming soon" candidates** — vote-gated films releasing within 90 days of
  `PREDICTION_DATE`, with a known runtime. They get the no-crowd model (§8).

`PREDICTION_DATE` is fixed rather than taken from today, so a rerun does not quietly change
the coming-soon set. The rules apply in order, so the exclusion counts do not overlap.

In [15]:
PREDICTION_DATE = pd.Timestamp("2026-09-21")
MIN_VOTES       = 7      # lowest vote count among the rated films
MAX_SHORT       = 40     # Academy definition of a short, in minutes

wl = wl_final.drop(columns=["vote_count"]).merge(wl_details, on="tmdb_id", how="left")
wl["release_date"] = pd.to_datetime(wl["release_date"], errors="coerce")

few_votes = wl["vote_count"] < MIN_VOTES
short     = ~few_votes & (wl["runtime"] <= MAX_SHORT)

upcoming = wl[few_votes
              & (wl["release_date"] > PREDICTION_DATE)
              & (wl["release_date"] <= PREDICTION_DATE + pd.Timedelta(days=90))
              & (wl["runtime"] > 0)].copy()

wl_pred = wl[~few_votes & ~short].copy()

out = (wl_pred[RANGE_COLS] < rated_lo) | (wl_pred[RANGE_COLS] > rated_hi)
wl_pred["out_of_range"] = out.apply(lambda row: "|".join(row.index[row]), axis=1)

print(f"matched films           : {len(wl)}")
print(f"excluded, < {MIN_VOTES} votes     : {few_votes.sum()}")
print(f"excluded, short (<= {MAX_SHORT}) : {short.sum()}")
print(f"to predict              : {len(wl_pred)}")
print(f"  of which flagged      : {(wl_pred['out_of_range'] != '').sum()}")
print(wl_pred.loc[wl_pred["out_of_range"] != "", "out_of_range"].value_counts().to_string())
print(f"\ncoming-soon candidates  : {len(upcoming)}")
print(upcoming[["tmdb_title", "release_date", "runtime"]]
      .sort_values("release_date").to_string(index=False))

matched films           : 4012
excluded, < 7 votes     : 45
excluded, short (<= 40) : 43
to predict              : 3924
  of which flagged      : 24
out_of_range
runtime         11
vote_average    11
popularity       2

coming-soon candidates  : 11
        tmdb_title release_date  runtime
     Possible Love   2026-09-23      165
         Primetime   2026-09-23      110
            Digger   2026-09-30      129
          Ray Gunn   2026-10-10      119
          Minotaur   2026-10-14      135
          Clayface   2026-10-21      108
   Wild Horse Nine   2026-11-04      118
       Paper Tiger   2026-11-12      115
         The Debut   2026-12-10      105
  Dune: Part Three   2026-12-15      140
Avengers: Doomsday   2026-12-16      165


**3,924 films to predict**, after 45 excluded by the vote gate and 43 as shorts; **24 flagged**
(11 runtime, 11 crowd score, 2 popularity). **11 coming-soon candidates** — including
*Primetime*, which has a few early votes but fewer than 7.

## 6. Features

History features for an unseen film: a watchlist film is a viewing that has not happened yet,
so it should see **all 1,192** rated viewings. The watchlist rows are appended **after** the
rated history with no rating, and the same functions run over both.

This exposed a bug in `genre_history`, which keeps running totals by hand: an unrated row added
NaN to every genre it touched, poisoning the total for every film after it. **Before the fix,
only 4 of 3,924 watchlist films got a genre mean.** The fix (in `src/features.py`) records the
row's features and then skips the update when the rating is missing — which also stops
watchlist films inflating each other's exposure counts. Rated data never has a missing rating,
so `04` is unaffected.

The check: rated rows must be **identical** with or without the watchlist appended, since
nothing later can change what came before.

In [16]:
rated_df = pd.read_csv("data/interim/modelling_base.csv", parse_dates=["watched_date"])
rated_df = rated_df.sort_values("watched_date").reset_index(drop=True)
GENRES_ALL = genre_vocabulary(rated_df["genres"])

combined = pd.concat([rated_df, wl_pred.assign(rating=np.nan)], ignore_index=True)
n = len(rated_df)

m_rated, x_rated = genre_history(rated_df, GENRES_ALL)
m_comb,  x_comb  = genre_history(combined, GENRES_ALL)

pd.testing.assert_series_equal(m_comb[:n], m_rated)
pd.testing.assert_series_equal(x_comb[:n], x_rated)
print(f"rated rows identical: {n} viewings")
print(f"genres in vocabulary: {len(GENRES_ALL)}")
print(f"watchlist rows with a genre mean: {m_comb[n:].notna().sum()} of {len(combined) - n}")

rated rows identical: 1192 viewings
genres in vocabulary: 16
watchlist rows with a genre mean: 3828 of 3924


Rated rows identical; **3,828 of 3,924** watchlist films now get a genre mean. The vocabulary,
re-derived from all 1,192 viewings, has **16 genres** — one more than `04`'s training split.

In [17]:
no_genre = combined[n:][m_comb[n:].isna()]
print(no_genre["genres"].fillna("(none)").value_counts().head(10).to_string())
print(f"\nvocabulary: {GENRES_ALL}")

genres
Documentary             68
Western                 26
Documentary|TV Movie     2

vocabulary: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'Thriller', 'War']


The 96 without one are **Documentary (70) and Western (26)** — neither reaches the 20-film
threshold for a column. Not a bug: they take the same imputation path as the 5 rated films
with no vocabulary genre. But their genre columns are all zero, a state seen only 5 times in
training, so they get a `genre` flag with the prediction.

### Assemble the matrices

- **Vocabularies from all 1,192 viewings** — the evaluated pipeline derived them from its own
  training data, so the refit does the same with its own.
- **`LOG_DEPLOY`** drops `review_words`, making these the deployable model's features.
- **Block H is built twice, deliberately.** Training rows come from the rated history alone,
  exactly as in `04` — building them from the combined frame would change the first film's
  median-imputed `hist_mean_all`. Watchlist rows come from the combined frame.
- **Keyword scores** are out-of-fold for training rows and full-fit for the watchlist.

In [18]:
LANGUAGES_ALL = top_n_vocabulary(rated_df["original_language"], n=10)
LOG_DEPLOY    = [c for c in LOG_NUMERIC if c != "review_words"]

wl_rows  = wl_pred.assign(rating=np.nan, film_decade=(wl_pred["film_year"] // 10) * 10)
combined = pd.concat([rated_df, wl_rows], ignore_index=True)

F_all   = build_features_F(combined, GENRES_ALL, LANGUAGES_ALL, PLAIN_NUMERIC, LOG_DEPLOY)
H_rated = build_features_H(rated_df, GENRES_ALL, HISTORY_KEYS)
H_comb  = build_features_H(combined, GENRES_ALL, HISTORY_KEYS)

kw_oof, kw_wl = keyword_scores(rated_df, wl_rows)

X_fit = add_keyword_score(pd.concat([F_all[:n], H_rated], axis=1), kw_oof)
X_wl  = add_keyword_score(pd.concat([F_all[n:], H_comb[n:]], axis=1).reset_index(drop=True), kw_wl)

print(f"languages: {LANGUAGES_ALL}")
print(f"X_fit {X_fit.shape}   X_wl {X_wl.shape}")
print(f"columns match: {list(X_fit.columns) == list(X_wl.columns)}")
print(f"nulls — fit {X_fit.isna().sum().sum()}, watchlist {X_wl.isna().sum().sum()}")
print(f"fit rows without a keyword score: {X_fit['kw_score_missing'].sum()}")
print(f"watchlist hist_count_all: {X_wl['hist_count_all'].unique()}")

languages: ['en', 'fr', 'ja', 'ko', 'zh']
X_fit (1192, 48)   X_wl (3924, 48)
columns match: True
nulls — fit 0, watchlist 0
fit rows without a keyword score: 112
watchlist hist_count_all: [1192.]


48 matching columns, no nulls. **112 training rows** have no keyword score — the first of eleven
time blocks takes the remainder (1,192 − 10 × 108), the same pattern as `04`'s 93 — and use the
usual fallback. Every watchlist row sees exactly 1,192 prior viewings.

## 7. Refit and predict

The deployable model refitted on all 1,192 viewings with the hyperparameters tuned in `04` §15,
**fixed** — no re-tuning on the full data, so the refit stays as close as possible to the model
that was evaluated.

In [19]:
FINAL_PARAMS = {"n_estimators": 300, "max_features": 0.5, "min_samples_leaf": 5, "random_state": 0}

final_rf = RandomForestRegressor(**FINAL_PARAMS).fit(X_fit, rated_df["rating"])

wl_out = wl_pred.reset_index(drop=True).copy()
wl_out["pred"] = final_rf.predict(X_wl)

genre_cols = [c for c in X_wl.columns if c.startswith("genre_")]
no_genre = X_wl[genre_cols].sum(axis=1) == 0
wl_out.loc[no_genre, "out_of_range"] = wl_out.loc[no_genre, "out_of_range"].apply(
    lambda s: "genre" if s == "" else s + "|genre")

print(wl_out["pred"].describe().round(2).to_string())
print(f"\ncorr with crowd score: {np.corrcoef(wl_out['pred'], wl_out['vote_average'])[0, 1]:.3f}")
print(f"flagged: {(wl_out['out_of_range'] != '').sum()}\n")

show = ["film_title", "film_year", "pred", "vote_average", "out_of_range"]
print("top 15:")
print(wl_out.nlargest(15, "pred")[show].round(2).to_string(index=False))
print("\nbottom 5:")
print(wl_out.nsmallest(5, "pred")[show].round(2).to_string(index=False))

count    3924.00
mean        3.57
std         0.43
min         2.14
25%         3.27
50%         3.60
75%         3.91
max         4.55

corr with crowd score: 0.645
flagged: 119

top 15:
                                 film_title  film_year  pred  vote_average out_of_range
The Human Condition III: A Soldier's Prayer     1961.0  4.55          8.42             
                             Doctor Zhivago     1965.0  4.55          7.60             
                        Fiddler on the Roof     1971.0  4.54          7.74             
   The Human Condition II: Road to Eternity     1959.0  4.53          8.22             
                                  Red Beard     1965.0  4.52          8.20             
             The Good, the Bad and the Ugly     1966.0  4.52          8.47        genre
                        Birdman of Alcatraz     1962.0  4.51          7.50             
               The Bridge on the River Kwai     1957.0  4.51          7.82             
                    

Predictions run **2.14 to 4.55** (mean 3.57), compressed as on the test set, with correlation
**0.645** against the crowd score. **119 flagged.**

The **top 15 is entirely 1953–1977** — long, canonical dramas and epics — and the bottom five are
2023–25 blockbusters, two of them with crowd scores around 6.5. The top is also flat: fifteen
films within 0.10 stars, so the app presents likely favourites rather than a strict order.

### Is the age pattern in the data?

A descriptive check, not a reason to change the model: actual ratings by decade against the
watchlist's mean prediction.

In [20]:
rated_by_dec = rated_df.groupby("film_decade")["rating"].agg(["count", "mean"])
wl_by_dec = (wl_out.assign(film_decade=(wl_out["film_year"] // 10) * 10)
             .groupby("film_decade")["pred"].agg(["count", "mean"]))

by_dec = rated_by_dec.join(wl_by_dec, lsuffix="_rated", rsuffix="_wl", how="outer")
print(by_dec.round(2).to_string())

             count_rated  mean_rated  count_wl  mean_wl
film_decade                                            
1910.0               NaN         NaN         5     3.61
1920.0               2.0        4.00        31     3.99
1930.0               6.0        3.58        62     3.76
1940.0              11.0        4.00        91     3.97
1950.0              24.0        4.15       173     4.00
1960.0              43.0        4.09       283     3.96
1970.0              55.0        4.21       378     3.86
1980.0              94.0        4.05       445     3.77
1990.0             135.0        3.84       586     3.61
2000.0             208.0        3.57       651     3.41
2010.0             277.0        3.53       802     3.35
2020.0             337.0        3.10       417     3.16


**It is in the data.** Actual ratings run **4.0–4.2 for the 1940s–80s** and fall steadily to
**3.10 for the 2020s** — and the counts show why: 43 rated 1960s films against 337 from the
2020s. Older films watched are the canon, pre-selected for quality; recent films are watched
more or less indiscriminately. The model has learned that faithfully, and predictions follow
the same gradient, shrunk toward the middle.

**Limitation:** the model cannot tell "old" from "pre-selected", because in the rated data they
never come apart. The watchlist reaches much deeper into each decade (283 1960s films against
43 rated), so **an average older film is likely over-predicted.**

The table also exposed an omission: `film_year` was not in the range check, and the watchlist
has five films from the 1910s, before any rated film.

In [21]:
yr_lo, yr_hi = rated_df["film_year"].min(), rated_df["film_year"].max()
off_year = (wl_out["film_year"] < yr_lo) | (wl_out["film_year"] > yr_hi)

wl_out.loc[off_year, "out_of_range"] = wl_out.loc[off_year, "out_of_range"].apply(
    lambda s: "film_year" if s == "" else s + "|film_year")

print(f"rated film_year range: {yr_lo:.0f}–{yr_hi:.0f}")
print(f"flagged on film_year: {off_year.sum()}")
print(f"total flagged: {(wl_out['out_of_range'] != '').sum()}")

rated film_year range: 1922–2026
flagged on film_year: 9
total flagged: 128


**9 flagged on `film_year`** — the five from the 1910s and four from 1920–21, before the earliest
rated film (1922). No overlap with existing flags: **128 flagged in total.**

### Gap above the crowd

The app's second sort. **Gap = prediction − Model 1's prediction.** Model 1 translates the crowd
score (TMDB, out of 10) into the viewer's own star scale, so the gap is in stars and means *how
much more than the crowd score alone would suggest*. A naive `vote_average / 2` would mix taste
with the difference between the two rating scales. The gap is validated on the test set in `04`
§17; Model 1 is refitted on all 1,192 viewings like the other models.

In [22]:
m1_final = LinearRegression().fit(rated_df[["vote_average"]], rated_df["rating"])

wl_out["crowd_pred"] = m1_final.predict(wl_out[["vote_average"]])
wl_out["gap"]        = wl_out["pred"] - wl_out["crowd_pred"]

print(f"Model 1 refit: slope {m1_final.coef_[0]:.3f}  intercept {m1_final.intercept_:.3f}\n")
print(wl_out["gap"].describe().round(2).to_string())

show = ["film_title", "film_year", "pred", "crowd_pred", "gap", "vote_average"]
print("\nbiggest gaps above the crowd:")
print(wl_out.nlargest(10, "gap")[show].round(2).to_string(index=False))
print("\nbiggest gaps below the crowd:")
print(wl_out.nsmallest(5, "gap")[show].round(2).to_string(index=False))

Model 1 refit: slope 0.573  intercept -0.535

count    3924.00
mean        0.22
std         0.34
min        -1.30
25%         0.04
50%         0.28
75%         0.44
max         1.84

biggest gaps above the crowd:
   film_title  film_year  pred  crowd_pred  gap  vote_average
The Conqueror     1956.0  3.37        1.53 1.84          3.60
        Gigli     2003.0  3.51        1.77 1.75          4.02
   Dracula 3D     2012.0  3.13        1.48 1.65          3.52
   Rollerball     2002.0  3.29        1.69 1.60          3.89
      Parents     2016.0  3.34        1.80 1.54          4.08
  The Canyons     2013.0  3.45        1.99 1.46          4.41
         3x3D     2013.0  3.45        2.04 1.41          4.50
        Hotel     2001.0  2.81        1.41 1.40          3.40
   Melissa P.     2005.0  3.55        2.15 1.40          4.69
       Ishtar     1987.0  3.38        2.08 1.31          4.56

biggest gaps below the crowd:
                 film_title  film_year  pred  crowd_pred   gap  vote_avera

Slope **0.573**, close to `04`'s 0.598. But the gap came out wrong at the extremes: mean +0.22,
range −1.30 to +1.84, and the top ten are **notorious flops** — *The Conqueror*, *Gigli*,
*Ishtar*. That is an artefact, not taste. Model 1 is a straight line that follows low crowd
scores all the way down — and extrapolates below the rated range: 6 of the ten score under the
lowest rated film's 4.29 — while the forest is compressed and never goes much below 2.1. Sorted
by this gap, the app would recommend films the crowd thinks are terrible.

The test-set validation covers mid-range crowd scores, not the extremes a sort selects. Two
rules, fixed before seeing their effect:

- **Rule 1 (data):** the gap is only computed where the crowd score lies within the **middle 95%
  of rated films' crowd scores** (2.5th–97.5th percentile). Elsewhere it is left empty; the film
  keeps its prediction.
- **Rule 2 (app):** the gap sort shows only films predicted at or above the viewer's average
  rating — recommendations the crowd undervalues, not films the viewer will merely dislike less
  than the crowd does. Applied in the app, so the saved gap stays a complete measurement.

In [23]:
crowd_lo, crowd_hi = rated_films["vote_average"].quantile([0.025, 0.975])
supported = wl_out["vote_average"].between(crowd_lo, crowd_hi)
wl_out.loc[~supported, "gap"] = np.nan

RATED_MEAN = rated_df["rating"].mean()

print(f"supported crowd range: {crowd_lo:.2f} – {crowd_hi:.2f}")
print(f"films with a gap: {supported.sum()} of {len(wl_out)}\n")

show = ["film_title", "film_year", "pred", "crowd_pred", "gap", "vote_average"]
print("biggest gaps above the crowd (rule 1):")
print(wl_out.nlargest(10, "gap")[show].round(2).to_string(index=False))

print(f"\nwhat the app's gap sort would show (rule 2: pred >= {RATED_MEAN:.2f}):")
print(wl_out[wl_out["pred"] >= RATED_MEAN].nlargest(10, "gap")[show].round(2).to_string(index=False))

print("\nbiggest gaps below the crowd:")
print(wl_out.nsmallest(5, "gap")[show].round(2).to_string(index=False))

supported crowd range: 5.71 – 8.30
films with a gap: 3663 of 3924

biggest gaps above the crowd (rule 1):
                     film_title  film_year  pred  crowd_pred  gap  vote_average
            Season of the Devil     2018.0  3.85        2.94 0.91          6.06
               October November     2013.0  3.73        2.86 0.87          5.92
                Ryan's Daughter     1970.0  4.39        3.52 0.87          7.08
                        Ragtime     1981.0  4.35        3.49 0.87          7.02
The Killing of a Chinese Bookie     1976.0  4.33        3.49 0.85          7.02
It's a Mad, Mad, Mad, Mad World     1963.0  4.29        3.46 0.83          6.98
                     The Castle     1997.0  3.77        2.95 0.83          6.08
                    The Captive     2000.0  3.58        2.75 0.82          5.74
              The Iceman Cometh     1973.0  3.76        2.94 0.82          6.06
                American Gigolo     1980.0  3.82        3.01 0.82          6.18

what the app'

Supported range **5.71–8.30**; **3,663 of 3,924** films keep a gap. The flops are gone. The top is
now films the crowd rates as middling — roughly 6–7 out of 10 — that the model puts well above
that: *Season of the Devil*, *The Killing of a Chinese Bookie*, *Ragtime*, *Ryan's Daughter*, *The
Captive*. Recognisably arthouse and New Hollywood the crowd undervalues, though still marked by
the age and length selection effects. Rule 2 changes nothing in the top ten — all predict at or
above 3.56, *The Captive* only just (3.58).

At the bottom: franchise and animation, plus *The Father* (8.09/10, predicted 2.99) — most likely
the recent-film pattern rather than anything specific to that film.

### Save

Only what the app needs to show and explain each film. `pred` is saved unrounded — rounding to
half-stars for display is the app's decision. `film_uri` links back to Letterboxd and is the one
column unique for every film, *The Castle* included. The matching audit columns stay in the
notebook. `crowd_pred` and `gap` are saved so the app can show both numbers with their units —
the crowd score out of 10, predictions in stars.

In [24]:
OUT_COLS = ["film_uri", "film_key", "film_title", "film_year", "tmdb_id",
            "pred", "crowd_pred", "gap", "vote_average", "vote_count", "runtime", "genres",
            "director", "original_language", "release_date", "poster_path", "out_of_range"]

PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

(wl_out[OUT_COLS]
 .sort_values("pred", ascending=False)
 .to_csv(PROCESSED / "watchlist_predictions.csv", index=False))

print(f"saved {len(wl_out)} films -> {PROCESSED / 'watchlist_predictions.csv'}")

saved 3924 films -> data/processed/watchlist_predictions.csv


### Why? — the rated films behind each prediction

A Random Forest prediction is an average of the ratings of training films sharing each tree's
leaf, so each rated film's **weight in the prediction** can be computed exactly: in every tree,
the films sharing the watchlist film's leaf each get 1 ÷ (films in that leaf), averaged over the
300 trees. The weights sum to 1, so the top ones are literally the films whose ratings produced
the prediction — the "Why?" step is exact, not illustrative. A separate content similarity
(shared genres, director) could explain a prediction with films that had nothing to do with it.

Rewatched films' weights are summed and shown with the most recent rating. The top 5 are saved
per film; **no review text** is saved — the app joins it from the uncommitted interim data.

**Check:** weighting the viewer's ratings this way should reconstruct the predictions closely —
not exactly, since each tree was trained on a bootstrap resample while the weights use every film.

In [25]:
N_NEIGHBOURS = 5

leaves_fit = final_rf.apply(X_fit)
leaves_wl  = final_rf.apply(X_wl)
n_trees    = leaves_fit.shape[1]

weight = np.zeros((len(X_wl), len(X_fit)), dtype=np.float32)
for t in range(n_trees):
    sizes = np.bincount(leaves_fit[:, t])
    weight += (leaves_wl[:, [t]] == leaves_fit[:, t]) / sizes[leaves_fit[:, t]]
weight /= n_trees

recon = weight @ rated_df["rating"].values
print(f"reconstructed vs actual prediction: mean |diff| {np.abs(recon - wl_out['pred']).mean():.3f}, "
      f"corr {np.corrcoef(recon, wl_out['pred'])[0, 1]:.3f}")

codes, film_keys = pd.factorize(rated_df["film_key"])
W = np.zeros((len(X_wl), len(film_keys)), dtype=np.float32)
for j, c in enumerate(codes):
    W[:, c] += weight[:, j]

films = (rated_df.drop_duplicates("film_key", keep="last")
         .set_index("film_key").loc[film_keys]
         .rename_axis("film_key").reset_index())

top = np.argsort(-W, axis=1)[:, :N_NEIGHBOURS]
neighbours = pd.DataFrame([
    {"film_uri": wl_out.at[i, "film_uri"], "rank": r + 1,
     "neighbour_key": films.at[j, "film_key"], "neighbour_title": films.at[j, "film_title"],
     "neighbour_year": films.at[j, "film_year"], "neighbour_rating": films.at[j, "rating"],
     "neighbour_tmdb_id": films.at[j, "tmdb_id"], "neighbour_poster": films.at[j, "poster_path"],
     "share": W[i, j]}
    for i, js in enumerate(top) for r, j in enumerate(js)
])

neighbours.to_csv(PROCESSED / "neighbours.csv", index=False)
print(f"saved {len(neighbours)} rows -> {PROCESSED / 'neighbours.csv'}\n")

example = wl_out["pred"].idxmax()
print(f"example: {wl_out.at[example, 'film_title']} — predicted {wl_out.at[example, 'pred']:.2f}")
print(neighbours[neighbours["film_uri"] == wl_out.at[example, "film_uri"]]
      [["rank", "neighbour_title", "neighbour_year", "neighbour_rating", "share"]]
      .round(3).to_string(index=False))

reconstructed vs actual prediction: mean |diff| 0.034, corr 0.998
saved 19620 rows -> data/processed/neighbours.csv

example: The Human Condition III: A Soldier's Prayer — predicted 4.55
 rank             neighbour_title  neighbour_year  neighbour_rating  share
    1               Andrei Rublev            1966               4.0  0.047
    2                         Ran            1985               4.5  0.035
    3                     Solaris            1972               4.5  0.034
    4       Judgment at Nuremberg            1961               4.5  0.033
    5 A Woman Under the Influence            1974               5.0  0.027


**Reconstruction: correlation 0.998, mean difference 0.034 stars** — the weights reflect what the
model does. The example reads as an explanation at a glance: *The Human Condition III* (4.55) is
treated most like *Andrei Rublev*, *Ran*, *Solaris*, *Judgment at Nuremberg* and *A Woman Under
the Influence*, all rated 4.0 or higher.

But the top five carry only **about 18%** of the prediction; the rest is spread across hundreds of
films in smaller amounts. They are the strongest influences, not the whole story — so the app
shows each neighbour's share.

## 8. Coming soon

Unreleased films have no crowd score, so they are scored by the **no-crowd variant** (`04` §16):
the deployable features minus `vote_average`, `vote_count` and `popularity`. On the test set it
scores **MAE 0.570**, beats the median by **+0.111 [0.053, 0.171]** — the pre-set condition for
using it at all — and even beats the crowd score without seeing it, **+0.086 [0.026, 0.143]**.
Not knowing the crowd costs **+0.025 [0.005, 0.046]** against the deployable model.

**That MAE is optimistic here.** It was measured on released films with complete TMDB data;
unreleased films have placeholder keywords and unsettled casts.

Two groups, one model:

- **From the watchlist** — the 11 candidates from §5, the real use case.
- **Popular upcoming releases** — for the demo. TMDB's global popularity turned out to be a poor
  selector: nearly flat below the top few, so regional bursts of attention decide most of the
  order. The list uses **films opening in UK cinemas** within the same 90 days, most popular
  first — what the audience can actually go and see.

The discover response is cached **per `PREDICTION_DATE` and region**, since TMDB's ranking
changes daily.

In [26]:
REGION = "GB"
UPCOMING_CACHE = Path(f"data/cache/upcoming_discover_{REGION}_{PREDICTION_DATE.date()}.json")
start = (PREDICTION_DATE + pd.Timedelta(days=1)).date().isoformat()
end   = (PREDICTION_DATE + pd.Timedelta(days=90)).date().isoformat()

if UPCOMING_CACHE.exists():
    discovered = json.loads(UPCOMING_CACHE.read_text())
else:
    discovered = discover_upcoming(start, end, headers=HEADERS, pages=2, region=REGION)
    UPCOMING_CACHE.write_text(json.dumps(discovered))

known = set(wl_final["tmdb_id"].astype(int)) | set(rated_films["tmdb_id"].astype(int))
popular = list({d["id"]: d for d in discovered if d["id"] not in known}.values())

print(f"window: {start} to {end}, {REGION} theatrical")
print(f"discovered {len(discovered)}, unique after removing watchlist and rated: {len(popular)}\n")
for d in popular:
    print(f"  {d['id']:>8}  {d.get('release_date', '????')}  {d['original_language']}  "
          f"{d.get('vote_count', 0):>5} votes  {d['title']}")

window: 2026-09-22 to 2026-12-20, GB theatrical
discovered 40, unique after removing watchlist and rated: 26

    299534  2019-04-25  en  28723 votes  Avengers: Endgame
   1263337  2026-09-19  en     12 votes  Heart of the Beast
    977942  2026-10-09  en     15 votes  The Uprising
   1465063  2026-10-04  en      8 votes  Forgotten Island
   1400837  2026-10-09  en      0 votes  Other Mommy
   1567937  2026-09-25  cn      6 votes  V
   1433583  2026-10-02  en     11 votes  The Weight
   1284046  2026-12-11  en     25 votes  Onslaught
   1300968  2026-11-20  en      0 votes  The Hunger Games: Sunrise on the Reaping
   1283515  2026-10-02  en      0 votes  Verity
   1153576  2026-10-16  en      0 votes  Street Fighter
    791523  2026-10-23  en      0 votes  Wildwood
     51739  2011-07-29  ja   3281 votes  Arrietty
   1397485  2026-10-02  en     65 votes  Girls Like Girls
   1281331  2026-10-09  en      0 votes  The Social Reckoning
   1381071  2026-11-03  ja      0 votes  Godzilla Minu

40 discovered, **26 unique** after removing films already on the watchlist or rated. Any repeated
film (the ranking can shift between page requests) is removed by ID — none this time. Two re-releases
got through the UK filter — *Avengers: Endgame* and *Arrietty* are back in cinemas — and are
removed below.

In [27]:
removed = [d for d in discovered if d["id"] in known]
print("removed as already on watchlist or rated:")
for d in removed:
    print(f"  {d['id']:>8}  {d.get('release_date', '????')}  {d['title']}")

removed as already on watchlist or rated:
   1058424  2026-09-25  Hope
   1003596  2026-12-18  Avengers: Doomsday
   1248832  2026-10-02  Digger
   1375441  2026-10-30  Primetime
   1170608  2026-12-18  Dune: Part Three
   1400940  2026-10-23  Clayface
   1437696  2026-10-23  Fatherland
   1469342  2026-09-25  Her Private Hell
   1483525  2026-10-23  Possible Love
   1401459  2026-11-13  Fjord
    891621  2026-11-05  Wild Horse Nine
     31767  1971-07-25  The Devils
    848700  2026-11-27  Minotaur
   1227877  2026-10-30  I Love Boosters


14 of the 40 are already on the watchlist or rated, including *Avengers: Doomsday* and *Dune:
Part Three* — they stay in the watchlist group, so no film is listed twice.

### Details and rules

- **Already in UK cinemas → excluded.** The displayed dates are UK dates; a re-release shows its
  original one, so any date before the window marks a film that has already been in cinemas.
- **The same runtime rules** as the watchlist: known and over 40 minutes.
- **Top 20 is a cap, not a target** — fewer survivors means a shorter list.

`uk_release` is kept because `parse_details` returns the *primary* release date, which for
international films differs from the UK one; `fetch_all_details` preserves order, so the dates
line up row for row.

In [28]:
UP_DETAILS_CACHE = Path(f"data/cache/upcoming_details_{REGION}_{PREDICTION_DATE.date()}.json")

already_out = [d for d in popular if (d.get("release_date") or "") < start]
popular     = [d for d in popular if (d.get("release_date") or "") >= start]
print("excluded, already in UK cinemas:")
for d in already_out:
    print(f"  {d.get('release_date', '????')}  {d['title']}")

popular_ids = [d["id"] for d in popular]
up_details = fetch_all_details(popular_ids, headers=HEADERS, cache_path=UP_DETAILS_CACHE)
up_details["popularity_rank"] = range(1, len(up_details) + 1)
up_details["uk_release"] = [d["release_date"] for d in popular]

ok = up_details["runtime"] > MAX_SHORT
popular_top = up_details[ok].head(20).copy()

print(f"\ncandidates          : {len(popular_ids)}")
print(f"failed runtime rules: {(~ok).sum()}")
print(f"kept                : {len(popular_top)}\n")
print(popular_top[["popularity_rank", "tmdb_title", "uk_release", "runtime",
                   "original_language", "genres"]].to_string(index=False))

excluded, already in UK cinemas:
  2019-04-25  Avengers: Endgame
  2026-09-19  Heart of the Beast
  2011-07-29  Arrietty
done — 0 new API calls, 23 from cache

candidates          : 23
failed runtime rules: 5
kept                : 18

 popularity_rank            tmdb_title uk_release  runtime original_language                                    genres
               1          The Uprising 2026-10-09      128                en                      Action|History|Drama
               2      Forgotten Island 2026-10-04      109                en Animation|Adventure|Fantasy|Comedy|Family
               3           Other Mommy 2026-10-09       93                en                                    Horror
               4                     V 2026-09-25      160                cn                               Drama|Crime
               5            The Weight 2026-10-02      112                en                      Drama|Thriller|Crime
               6             Onslaught 2026-12-11  

**18 kept.** The five runtime failures include the most recognisable titles — *The Hunger Games:
Sunrise on the Reaping*, *Ramayana*, *The Further Mis-Adventures of Cliff Booth*, *The Social
Reckoning*, *How to Rob a Bank* — because TMDB has not published their runtimes. **The rule
stays:** it was fixed before this list was seen, and TMDB's 0 would sit far below any runtime in
training. The honest route to include them is a later `PREDICTION_DATE`, once runtimes are
published — the caches are keyed by date, so the whole pipeline reruns cleanly on fresher data.

### Predict

Both groups go through the same feature path as the watchlist — Block F, Block H appended after
the full history, keyword scores from all rated films — and then drop the three crowd columns;
the `assert` guarantees the columns match training exactly. The no-crowd model is refitted on
all 1,192 viewings with the hyperparameters tuned in `04` §16 (`min_samples_leaf` 15).

`release_shown` is the date the app displays: the primary date for watchlist films, the UK
cinema date for popular ones.

In [29]:
CROWD          = ["vote_average", "log_vote_count", "log_popularity"]
NOCROWD_PARAMS = {"n_estimators": 300, "max_features": 0.5, "min_samples_leaf": 15, "random_state": 0}

cs_wl = upcoming.assign(source="watchlist",
                        release_shown=upcoming["release_date"].dt.date.astype(str))
cs_pop = popular_top.assign(source="popular",
                            film_title=popular_top["tmdb_title"],
                            film_year=pd.to_datetime(popular_top["release_date"]).dt.year,
                            film_uri=None,
                            release_shown=popular_top["uk_release"])
cs_pop["film_key"] = cs_pop["film_title"] + " (" + cs_pop["film_year"].astype(str) + ")"

cs = pd.concat([cs_wl, cs_pop], ignore_index=True)
cs_rows = cs.assign(rating=np.nan, film_decade=(cs["film_year"] // 10) * 10)

F_cs = build_features_F(cs_rows, GENRES_ALL, LANGUAGES_ALL, PLAIN_NUMERIC, LOG_DEPLOY)
H_cs = build_features_H(pd.concat([rated_df, cs_rows], ignore_index=True),
                        GENRES_ALL, HISTORY_KEYS)[n:].reset_index(drop=True)
_, kw_cs = keyword_scores(rated_df, cs_rows)

X_cs     = add_keyword_score(pd.concat([F_cs, H_cs], axis=1), kw_cs).drop(columns=CROWD)
X_fit_nc = X_fit.drop(columns=CROWD)
assert list(X_cs.columns) == list(X_fit_nc.columns)

nocrowd_rf = RandomForestRegressor(**NOCROWD_PARAMS).fit(X_fit_nc, rated_df["rating"])
cs["pred"] = nocrowd_rf.predict(X_cs)

rng_cols = ["runtime", "film_year"]
off = (cs[rng_cols] < rated_df[rng_cols].min()) | (cs[rng_cols] > rated_df[rng_cols].max())
off["genre"] = X_cs[genre_cols].sum(axis=1).values == 0
cs["out_of_range"] = off.apply(lambda r: "|".join(r.index[r]), axis=1)

CS_COLS = ["source", "film_uri", "film_key", "film_title", "film_year", "tmdb_id", "pred",
           "release_shown", "runtime", "genres", "director", "original_language",
           "poster_path", "out_of_range"]
cs = cs.sort_values(["source", "pred"], ascending=[False, False])
cs[CS_COLS].to_csv(PROCESSED / "coming_soon.csv", index=False)

print(f"saved {len(cs)} films -> {PROCESSED / 'coming_soon.csv'}\n")
print(cs[["source", "film_title", "release_shown", "pred", "out_of_range"]]
      .round(2).to_string(index=False))

saved 29 films -> data/processed/coming_soon.csv

   source            film_title release_shown  pred out_of_range
watchlist         Possible Love    2026-09-23  3.95             
watchlist              Minotaur    2026-10-14  3.79             
watchlist              Ray Gunn    2026-10-10  3.46             
watchlist           Paper Tiger    2026-11-12  3.06             
watchlist      Dune: Part Three    2026-12-15  3.03             
watchlist             Primetime    2026-09-23  2.95             
watchlist                Digger    2026-09-30  2.89             
watchlist             The Debut    2026-12-10  2.78             
watchlist    Avengers: Doomsday    2026-12-16  2.77             
watchlist       Wild Horse Nine    2026-11-04  2.73             
watchlist              Clayface    2026-10-21  2.72             
  popular         La bola negra    2026-11-06  3.63             
  popular          The Paradise    2026-09-23  3.58             
  popular                     V    2026-

**Lower on average than the main watchlist**, as expected for 2026 films (rated 2020s films
average 3.10), and blockbusters sit at the bottom — *Focker-in-Law* 2.44, *Street Fighter* 2.52,
*Avengers: Doomsday* 2.77.

The top of both groups shows the model's other selection effect: **long, non-English films** —
*Possible Love* (Korean, 165 min), *La bola negra* (Spanish, 155), *V* (Cantonese, 160),
*The Paradise* (Telugu, 174). The non-English films the viewer has rated were sought out, so
the model learned "non-English and long means liked". *The Paradise*, a three-hour Telugu action
film at 3.58, is almost certainly that signal rather than taste — and without the crowd score to
anchor it, the no-crowd model leans on it harder. The same limitation as the decade pattern:
the model learns from **what was chosen**, not only from how it was rated.

---

## Summary

| | |
|---|---|
| Watchlist films | 4,031 → 4,012 matched → **3,924 predicted** |
| Excluded | 8 undated, 11 TV / not on TMDB, 45 under 7 votes, 43 shorts |
| Resolved by hand | 5 overrides, 2 collisions by URI |
| Flagged, still predicted | 128 (runtime, crowd score, popularity, year, no vocabulary genre) |
| Coming soon | 11 from the watchlist + 18 popular UK releases |
| Models | Deployable (MAE 0.545) · no-crowd (MAE 0.570) — both refitted on 1,192 viewings |
| Gap above the crowd | 3,663 films (crowd score 5.71–8.30) — validated in `04` §17 |
| Why? | Top 5 rated films per prediction by weight (reconstruction r = 0.998) |

**Known limitations**

- **Selection effects.** Older and non-English films in the rated history were pre-selected, so
  the model rewards age and foreign-language status beyond what taste alone would; an average
  film of either kind is likely over-predicted.
- **Predictions are compressed** — nothing above 4.55 — and the top of the ranking is flat.
- **The gap is only defined mid-range.** Outside the middle 95% of rated crowd scores, a
  straight-line Model 1 and the compressed forest diverge mechanically.
- **Unreleased films** are scored with less information, and their TMDB data is still changing;
  the most anticipated titles may lack runtimes until close to release.
- **TMDB values are fetched once, at `PREDICTION_DATE`.** For the watchlist that is the right
  moment; refreshing means changing the date, not re-running silently.

**Next:** the Streamlit app, reading `data/processed/`.